<a href="https://colab.research.google.com/github/davimonteirox/CHECKPOINT2-SERS/blob/main/Untitled1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
### MARKDOWN ###
# # Checkpoint 02 - APIs de Energia Renovável e Aprendizado de Máquina
# **Nome do Aluno:** [DAVI MONTEIRO]

# ## Parte Inicial: Extração dos Dados via APIs (Fornecido no Enunciado)

In [ ]:
### CÉLULA DE CÓDIGO ###
import json
import urllib.parse
import urllib.request
import csv

def consultar_api(endereco, parametros):
   url = endereco + "?" + urllib.parse.urlencode(parametros)
   try:
       with urllib.request.urlopen(url, timeout=30) as resposta:
           return json.load(resposta)
   except Exception as erro:
       raise RuntimeError(f"Não foi possível consultar a API: {erro}") from erro

# 1. Consultar a ANEEL
URL_ANEEL = "https://dadosabertos.aneel.gov.br/api/3/action/datastore_search"
ID_RECURSO = "11ec447d-698d-4ab8-977f-b424d5deee6a"
SIGLAS = ["UFV", "EOL", "UHE", "PCH", "CGH"]
print("API ANEEL: consulta pública, sem token")

registros_aneel = []
for sigla in SIGLAS:
   parametros = {
       "resource_id": ID_RECURSO,
       "filters": json.dumps({"SigTipoGeracao": sigla}),
       "limit": 1200,
   }
   resposta = consultar_api(URL_ANEEL, parametros)
   if not resposta.get("success"):
       raise ValueError(f"Consulta da ANEEL falhou para {sigla}")
   linhas = resposta["result"]["records"]
   print(f"{sigla}: {len(linhas)} linhas recebidas")
   registros_aneel.extend(linhas)
print("Total de linhas recebidas:", len(registros_aneel))

# 2. Gerar o CSV para a tarefa 1
def numero(valor):
   texto = str(valor).strip()
   if "," in texto:
       texto = texto.replace(".", "").replace(",", ".")
   try:
       return float(texto)
   except ValueError:
       return None

fontes = {"UFV": "Solar", "EOL": "Eólica", "UHE": "Hidráulica",
         "PCH": "Hidráulica", "CGH": "Hidráulica"}
linhas_classificacao = []
for registro in registros_aneel:
   potencia = numero(registro.get("MdaPotenciaOutorgadaKw"))
   latitude = numero(registro.get("NumCoordNEmpreendimento"))
   longitude = numero(registro.get("NumCoordEEmpreendimento"))
   fonte = fontes.get(registro.get("SigTipoGeracao"))
   if fonte and all(valor is not None for valor in (potencia, latitude, longitude)):
       linhas_classificacao.append([potencia, latitude, longitude, fonte])

with open("aneel_classificacao_orange.csv", "w", newline="", encoding="utf-8-sig") as arquivo:
   escritor = csv.writer(arquivo)
   escritor.writerow(["potencia_kw", "latitude", "longitude", "fonte"])
   escritor.writerows(linhas_classificacao)
print("Salvo: aneel_classificacao_orange.csv")

# 3. Consultar Open-Meteo
URL_METEO = "https://archive-api.open-meteo.com/v1/archive"
parametros_meteo = {
   "latitude": -9.39,
   "longitude": -40.50,
   "start_date": "2025-04-01",
   "end_date": "2025-06-30",
   "timezone": "America/Recife",
   "hourly": ",".join(["temperature_2m", "relative_humidity_2m",
                        "cloud_cover", "wind_speed_10m", "shortwave_radiation"]),
}
print("\nOpen-Meteo: consulta pública, sem token")

resposta_meteo = consultar_api(URL_METEO, parametros_meteo)
horas_api = resposta_meteo["hourly"]

# 4. Gerar CSV Horário
nomes_api = ["temperature_2m", "relative_humidity_2m", "cloud_cover",
            "wind_speed_10m", "shortwave_radiation"]
linhas_regressao = []
for indice, data_hora in enumerate(horas_api["time"]):
   hora = int(data_hora[11:13])
   valores = [horas_api[nome][indice] for nome in nomes_api]
   if 7 <= hora <= 17 and all(valor is not None for valor in valores):
       linhas_regressao.append([data_hora, *valores[:4], hora, valores[4]])

with open("meteo_regressao_orange.csv", "w", newline="", encoding="utf-8-sig") as arquivo:
   escritor = csv.writer(arquivo)
   escritor.writerow(["data_hora", "temperatura_c", "umidade_pct",
                      "nuvens_pct", "vento_kmh", "hora", "radiacao_w_m2"])
   escritor.writerows(linhas_regressao)
print("Salvo: meteo_regressao_orange.csv")

In [ ]:
### MARKDOWN ###
# ---
# ## Tarefa 1 — Classificação (ANEEL)
# **Objetivo:** Classificar a fonte de energia (Solar, Eólica ou Hidráulica) com base na potência, latitude e longitude.

In [ ]:
### CÉLULA DE CÓDIGO ###
# Imports para análise e Machine Learning
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score, precision_score, recall_score, f1_score

# 1. Carregando e Analisando os Dados
df_aneel = pd.read_csv("aneel_classificacao_orange.csv")
display(df_aneel.head())
display(df_aneel.info())
display(df_aneel['fonte'].value_counts())

# Gráfico de dispersão (Localização por Fonte)
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_aneel, x='longitude', y='latitude', hue='fonte', alpha=0.6)
plt.title("Distribuição Geográfica dos Empreendimentos por Fonte")
plt.show()

# 2. Preparando os Dados (Divisão e Escalonamento)
X_class = df_aneel[['potencia_kw', 'latitude', 'longitude']]
y_class = df_aneel['fonte']

# Divisão estratificada (80/20) com semente fixa
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_class, y_class, test_size=0.2, random_state=42, stratify=y_class)

# Escalonamento (fit apenas no treino)
scaler_c = StandardScaler()
X_train_c_scaled = scaler_c.fit_transform(X_train_c)
X_test_c_scaled = scaler_c.transform(X_test_c)

# 3. Treinando os Modelos
modelos_class = {
   "Random Forest": RandomForestClassifier(random_state=42),
   "SVM": SVC(random_state=42),
   "KNN": KNeighborsClassifier()
}

resultados_class = []

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (nome, modelo) in enumerate(modelos_class.items()):
   # Treino
   modelo.fit(X_train_c_scaled, y_train_c)
   # Previsão
   y_pred = modelo.predict(X_test_c_scaled)

   # Métricas (Média Macro)
   acc = accuracy_score(y_test_c, y_pred)
   prec = precision_score(y_test_c, y_pred, average='macro', zero_division=0)
   rec = recall_score(y_test_c, y_pred, average='macro', zero_division=0)
   f1 = f1_score(y_test_c, y_pred, average='macro', zero_division=0)

   resultados_class.append({"Modelo": nome, "Accuracy": acc, "Precision (Macro)": prec, "Recall (Macro)": rec, "F1-Score (Macro)": f1})

   # Matriz de Confusão
   cm = confusion_matrix(y_test_c, y_pred, labels=modelo.classes_)
   disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=modelo.classes_)
   disp.plot(ax=axes[idx], cmap='Blues', colorbar=False)
   axes[idx].set_title(f"Matriz de Confusão - {nome}")

plt.tight_layout()
plt.show()

# Tabela de Resultados
df_res_class = pd.DataFrame(resultados_class)
display(df_res_class)

In [ ]:
### MARKDOWN ###
# ### Conclusões - Classificação
#
# **1. Variáveis X e Y:** As variáveis independentes ($X$) são as características físicas e geográficas do empreendimento (`potencia_kw`, `latitude`, `longitude`). A variável dependente ($y$) é a categoria da fonte geradora (`fonte`).
#
# **2. Escolha do Modelo e Confusões:**
# Analisando a tabela de métricas e as matrizes de confusão, o **Random Forest** costuma apresentar o melhor desempenho geral (maior F1-Score Macro), pois lida muito bem com relações não-lineares (como a relação entre localização geográfica e tipo de clima favorável a certa energia). As classes mais confundidas costumam ser **Solar e Eólica** ou **Solar e Hidráulica** dependendo da região, devido ao desequilíbrio de classes (a classe Solar UFV tem muitos registros, podendo enviesar modelos mais simples).
#
# **3. Por que potência e localização não bastam para uma aplicação real?**
# Apenas saber onde um terreno está e qual a potência da usina não define a tecnologia. Um mesmo local geográfico pode abrigar um parque eólico, uma fazenda solar ou ambos (projetos híbridos). Além disso, a potência nominal de uma pequena usina hidrelétrica (PCH) pode ser idêntica à de uma fazenda solar média. Para uma classificação precisa no mundo real, seriam necessários dados como área ocupada, investimento, características do relevo ou licenças ambientais específicas.

In [ ]:
### MARKDOWN ###
# ---
# ## Tarefa 2 — Regressão (Open-Meteo)
# **Objetivo:** Estimar a radiação solar em Petrolina (PE) com base no clima e na hora do dia.

In [ ]:
### CÉLULA DE CÓDIGO ###
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Carregando e Analisando
df_meteo = pd.read_csv("meteo_regressao_orange.csv")
df_meteo['data_hora'] = pd.to_datetime(df_meteo['data_hora']) # Garantindo tipo datetime
df_meteo = df_meteo.sort_values(by='data_hora') # Garantindo ordem cronológica

display(df_meteo.head())
display(df_meteo.info())

# Visualização da radiação em relação à hora do dia
plt.figure(figsize=(8, 5))
sns.boxplot(data=df_meteo, x='hora', y='radiacao_w_m2')
plt.title("Distribuição da Radiação Solar por Hora do Dia (Petrolina-PE)")
plt.show()

# 2. Preparando os Dados (Divisão Temporal)
X_reg = df_meteo[['temperatura_c', 'umidade_pct', 'nuvens_pct', 'vento_kmh', 'hora']]
y_reg = df_meteo['radiacao_w_m2']

# 80% treino, 20% teste (SEM SHUFFLE para manter ordem temporal)
split_idx = int(len(df_meteo) * 0.8)

X_train_r = X_reg.iloc[:split_idx]
X_test_r = X_reg.iloc[split_idx:]
y_train_r = y_reg.iloc[:split_idx]
y_test_r = y_reg.iloc[split_idx:]

# Escalonamento
scaler_r = StandardScaler()
X_train_r_scaled = scaler_r.fit_transform(X_train_r)
X_test_r_scaled = scaler_r.transform(X_test_r)

# 3. Treinando os Modelos
modelos_reg = {
   "Regressão Linear": LinearRegression(),
   "Random Forest Regressor": RandomForestRegressor(random_state=42),
   "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}

resultados_reg = []
previsoes_dict = {}

for nome, modelo in modelos_reg.items():
   # Treino
   modelo.fit(X_train_r_scaled, y_train_r)
   # Previsão
   y_pred = modelo.predict(X_test_r_scaled)
   previsoes_dict[nome] = y_pred

   # Métricas
   mae = mean_absolute_error(y_test_r, y_pred)
   mse = mean_squared_error(y_test_r, y_pred)
   r2 = r2_score(y_test_r, y_pred)

   resultados_reg.append({"Modelo": nome, "MAE (W/m²)": mae, "MSE (W/m²)²": mse, "R²": r2})

# Tabela de Resultados
df_res_reg = pd.DataFrame(resultados_reg)
display(df_res_reg)

# 4. Gráfico de Valores Reais vs Previstos (Melhor modelo - Ex: Random Forest)
plt.figure(figsize=(12, 6))
# Plotando apenas uma amostra (ex: primeiros 100 registros do teste) para melhor visualização
amostra = 100
plt.plot(df_meteo['data_hora'].iloc[split_idx:split_idx+amostra], y_test_r.iloc[:amostra], label='Real (W/m²)', marker='o')
plt.plot(df_meteo['data_hora'].iloc[split_idx:split_idx+amostra], previsoes_dict["Random Forest Regressor"][:amostra], label='Previsto - RF', linestyle='--', marker='x')
plt.title("Radiação Solar: Real vs Previsto (Amostra do Teste)")
plt.xlabel("Data/Hora")
plt.ylabel("Radiação Solar (W/m²)")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
### MARKDOWN ###
# ### Conclusões - Regressão
#
# **1. Interpretação dos Erros e Modelos:**
# A **Regressão Linear** apresenta o pior desempenho (menor R²), pois a relação entre radiação e hora do dia forma uma curva (parábola), e não uma linha reta. Modelos baseados em árvores (como **Random Forest** e **Gradient Boosting**) conseguem capturar essa não-linearidade perfeitamente, apresentando os menores MAE (Erro Médio Absoluto) e os maiores R².
#
# **2. O peso da variável "hora":**
# Conforme visto no gráfico de boxplot da Análise Exploratória, a hora do dia é a variável preditora mais importante. A radiação solar segue um ciclo natural, nascendo por volta das 7h, atingindo o pico entre 11h e 13h, e caindo no fim da tarde. Nuvens e chuva apenas aplicam um "desconto" nesse padrão base ditado pela hora (e pela posição do sol).
#
# **3. Por que Radiação (W/m²) não é Energia (kWh)?**
# O modelo prevê a radiação que atinge o solo. No entanto, para calcular a *energia produzida* por uma usina, precisaríamos saber:
# * A área total (tamanho) dos painéis solares.
# * A eficiência técnica do painel (geralmente entre 15% e 22%, ou seja, a maior parte da radiação é perdida).
# * Perdas nos inversores, cabos e degradação por temperatura (painéis muito quentes perdem eficiência).
# Portanto, a radiação é apenas o "combustível" bruto, não a energia líquida injetada na rede.